# Lab 5.2 &mdash; Build a Tool: Give the Agent Hands

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Represent a tool as {name, description, fn} and build a registry
- Implement a SAFE calculator (no bare eval) and a lookup tool
- Let the REAL model pick a tool by reading your descriptions

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; Tools turn an LLM into an agent](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = "/tmp/biaa-lab-05-02"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
A model alone only emits **text**. A **tool** is a plain function you expose to it, described by a
**name**, a **description** (which the model reads to decide when to use it) and the **function** itself.
Two staples: a **calculator** (LLMs are bad at exact math &mdash; offload it) and a **lookup**. The
description is the tool's real API: below, the **real model** picks a tool purely from your words.

> **Safety:** never run bare `eval()` on free text. Our calculator uses a tiny **AST-based** safe
> evaluator that only allows numbers and arithmetic operators.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)
print("safe_calc('2+2') =", safe_calc("2+2"), "| safe_calc('3*(4+1)') =", safe_calc("3*(4+1)"))
# A bare eval() would happily run "__import__('os').system('rm -rf /')" -- ours refuses.

## Build it
Implement the two tool functions, then build the registry mapping each tool's name to its dict.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

KNOWLEDGE = {"capital of france": "Paris", "population of metropolis": "120000"}

def calculator_fn(expr):
    return ___    # TODO: evaluate expr with the safe calculator (never bare eval)

def lookup_fn(key):
    return ___    # TODO: look key up in KNOWLEDGE (lowercased/stripped), default "unknown"

calculator = {"name": "calculator", "description": "Do exact arithmetic like 2+2 or 45*12.", "fn": calculator_fn}
lookup     = {"name": "lookup", "description": "Look up a known fact by key, e.g. capital of france.", "fn": lookup_fn}

def build_registry(tools):
    return ___    # TODO: a dict mapping each tool's name to the tool

try:
    REGISTRY = build_registry([calculator, lookup])
    print("tools:", list(REGISTRY))
    print("calculator(45*12) =", REGISTRY["calculator"]["fn"]("45*12"))
    print("lookup(capital of france) =", REGISTRY["lookup"]["fn"]("capital of france"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Show the real model the tool catalog and let *it* pick which tool answers a question &mdash; from the descriptions alone.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        catalog = "\n".join("- " + t["name"] + ": " + t["description"] for t in REGISTRY.values())
        question = "what is 45 * 12?"
        prompt = ("You can use these tools:\n" + catalog +
                  "\n\nWhich ONE tool answers this question: '" + question + "'?\n"
                  "Reply with ONLY the tool name.")
        choice = llm_text(prompt).strip().lower()
        print("model read the descriptions and picked:", repr(choice))
        name = next((n for n in REGISTRY if n in choice), None)   # match its pick to a REAL tool, safely
        if name:
            print("running", name, "->", REGISTRY[name]["fn"]("45*12"))
        else:
            print("(model didn't name a known tool -- that happens; sharpen the descriptions and retry.)")
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The model chose a tool by reading the **description** &mdash; it never saw your code. The docstring/description **is** the tool's API.
- We matched its free-text pick back to a real tool **safely** (`next(... , None)`) &mdash; if it names nothing valid, we say so instead of crashing.
- In Module 6 a framework wires this selection for you; here you saw the raw mechanism.

## Your turn (open task &mdash; no grader)
Add a **third** tool (e.g. a `weather` tool) with a clear description, rebuild the registry, and re-ask the
model with a weather question. **What good looks like:** the model picks your new tool from its description.
Make the description vague and watch it pick worse &mdash; proof that you write descriptions *for the model*.

---
### Key takeaway
A tool is `{name, description, fn}` in a registry, and the description is what a REAL model reads to choose it. Next: give the agent a loop that calls the tool the model picks.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>